In [18]:
##### Re-scales country-level and global percentile rasters for relative deprivation to be population weighted 
# based on the CIESIN GRDI relative deprivation index - download from: https://www.earthdata.nasa.gov/data/catalog/sedac-ciesin-sedac-pmp-grdi-2010-2020-1.00
# also requires population dataset from Bondarenko et al (2025) - download from: https://doi.org/10.5258/SOTON/WP00846

import numpy as np
import sys
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
from rasterio.warp import Resampling
from rasterio import warp as rio_warp
import rasterio.features
from scipy.spatial import cKDTree
from scipy.ndimage import find_objects

In [19]:
##### Load data
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from helpers.reproject_raster import reproject_raster, array_to_raster

# set directories
DATA_DIR = Path(ROOT / "Data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path(ROOT / "Results" / "30x30")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# country geographies
countries = gpd.read_file(DATA_DIR / "gadm_410-levels.gpkg", layer="ADM_0")

# relative deprivation raster
relative_deprevation_path = DATA_DIR / "povmap-grdi-v1.tif"

# population raster used as weight 
population_path = DATA_DIR / "global_pop_2020_CN_1km_R2025A_UA_v1.tif"

# conservation scenario raster (as reference)
biodiversity_with_PA = DATA_DIR / "conservation_scenario.tif"

In [20]:
##### DATA PREP

# resample to match conservation scenario (equal area projection)
def load_summed(path):
    return array_to_raster(reproject_raster(str(path), resampling="mean"))
relative_deprevation = load_summed(relative_deprevation_path)

def load_summed(path):
    return array_to_raster(reproject_raster(str(path), resampling="sum"))
population = load_summed(population_path)

In [21]:
##### FILL MISSING RELATIVE DEPRIVATION DATA

# reference grid 
ref = relative_deprevation
ref_transform = ref.rio.transform()
ref_shape = (ref.rio.height, ref.rio.width)

# 1. country ID raster (pixel centroid inside country i -> i, else -1)
countries_proj = countries.to_crs(ref.rio.crs).reset_index(drop=True)

country_id_arr = rasterio.features.rasterize(
    shapes=((geom, i) for i, geom in enumerate(countries_proj.geometry) if geom is not None),
    out_shape=ref_shape,
    transform=ref_transform,
    fill=-1,
    dtype="int32",
    all_touched=False,   
)

# pixels whose centroid is outside every polygon but which are touched by one (coastal pixels)
country_id_touched = rasterio.features.rasterize(
    shapes=((geom, i) for i, geom in enumerate(countries_proj.geometry) if geom is not None),
    out_shape=ref_shape,
    transform=ref_transform,
    fill=-1,
    dtype="int32",
    all_touched=True,
)
country_id_arr = np.where(country_id_arr < 0, country_id_touched, country_id_arr)

# 2. arrays
rd = ref.squeeze().values.astype(float)
pop = population.squeeze().values.astype(float)
assert rd.shape == pop.shape == country_id_arr.shape

in_country = country_id_arr >= 0
populated = np.nan_to_num(pop, nan=0) > 0
missing = np.isnan(rd)

to_fill = in_country & populated & missing   
valid_src = in_country & ~missing            

# 3. pixel coordinates 
def rowcol_to_xy(rows, cols):
    x = ref_transform.c + (cols + 0.5) * ref_transform.a
    y = ref_transform.f + (rows + 0.5) * ref_transform.e
    return np.column_stack([x, y])

# 4. group pixel indices by country  
def group_by_country(mask):
    flat_idx = np.flatnonzero(mask)
    ids = country_id_arr.ravel()[flat_idx]
    order = np.argsort(ids, kind="stable")
    flat_idx, ids = flat_idx[order], ids[order]
    uniq, start = np.unique(ids, return_index=True)
    return dict(zip(uniq, np.split(flat_idx, start[1:])))

src_by_country = group_by_country(valid_src)
fill_by_country = group_by_country(to_fill)

# 5. nearest-neighbour fill within each country
rd_flat = rd.ravel().copy()
filled_flag = np.zeros(rd.size, dtype=bool)
no_source = []   # countries with pixels to fill but no valid pixels to copy from

for cid, fill_idx in fill_by_country.items():
    src_idx = src_by_country.get(cid)
    if src_idx is None or len(src_idx) == 0:
        no_source.append(countries_proj.loc[cid, "GID_0"])
        continue

    src_xy = rowcol_to_xy(*np.unravel_index(src_idx, ref_shape))
    fill_xy = rowcol_to_xy(*np.unravel_index(fill_idx, ref_shape))

    _, nn = cKDTree(src_xy).query(fill_xy, k=1)
    rd_flat[fill_idx] = rd_flat[src_idx[nn]]
    filled_flag[fill_idx] = True

rd_filled = rd_flat.reshape(rd.shape)
filled_flag = filled_flag.reshape(rd.shape)

# 6. summary
print(f"Pixels needing fill:     {to_fill.sum():,}")
print(f"Pixels filled:           {filled_flag.sum():,}")
print(f"Still missing (no valid pixel in same country): {(to_fill & ~filled_flag).sum():,}")
if no_source:
    print(f"Countries with no source pixels: {no_source}")

# 7. back to DataArray 
relative_deprevation_filled = ref.copy(data=rd_filled[np.newaxis, ...] if ref.ndim == 3 else rd_filled)

Pixels needing fill:     291,242
Pixels filled:           290,957
Still missing (no valid pixel in same country): 285
Countries with no source pixels: ['BLM', 'BMU', 'COK', 'CUW', 'FLK', 'JEY', 'MAF', 'NIU', 'SXM', 'TKL']


In [22]:
##### Prep & helpers

# inputs as plain arrays on the common grid
value_arr = rd_filled                       # relative deprivation after nearest-neighbour fill
pop_arr = pop                               # population (sum-resampled)
assert value_arr.shape == pop_arr.shape == country_id_arr.shape

# limit percentiles to pixels assigned to a country
land_mask = country_id_arr >= 0

ref_crs = ref.rio.crs

# helpers
def save_raster(arr, out_path):
    meta = dict(
        driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1,
        dtype="float32", crs=ref_crs, transform=ref_transform, nodata=np.nan,
        compress="deflate",
    )
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr.astype(np.float32), 1)

# population-weighted CDF
def weighted_percentile_lookup(values, weights, invert=False):
    unique_vals, inverse = np.unique(values, return_inverse=True)
    pop_per_val = np.zeros(len(unique_vals))
    np.add.at(pop_per_val, inverse, weights)

    cum_pop = np.cumsum(pop_per_val)
    midpoint_share = (cum_pop - pop_per_val / 2.0) / cum_pop[-1]
    pct_int = np.clip(np.ceil(midpoint_share * 100), 1, 100)

    if invert:
        pct_int = 101 - pct_int   # highest value -> low percentile, lowest value -> 100
    return unique_vals, pct_int

def assign_percentiles(sub_values, unique_vals, pct_int):
    assigned = np.interp(sub_values, unique_vals, pct_int)
    return np.clip(np.round(assigned), 1, 100).astype(np.float32)

valid_val = ~np.isnan(value_arr) & land_mask
has_pop_weight = ~np.isnan(pop_arr) & (pop_arr > 0)

In [23]:
##### Step 1: country-level percentiles

def percentile_raster_by_country(value_arr, pop_arr, country_id_arr, valid_val, has_pop_weight, invert=False):
    """
    Population-weighted, within-country percentile of value_arr.
    - Pixels with pop > 0 and a valid value build each country's CDF.
    - All valid-value land pixels (including pop == 0) get a percentile from that CDF.
    - Countries with no populated valid pixels are skipped (their pixels stay NaN).
    """
    out = np.full(value_arr.shape, np.nan, dtype=np.float32)

    # bounding box per country (labels shifted by +1 because find_objects ignores 0)
    slices = find_objects(country_id_arr + 1)

    for label_minus_1, sl in enumerate(slices):
        if sl is None:
            continue
        cid = label_minus_1   # country_id_arr value

        sub_mask = country_id_arr[sl] == cid
        sub_value = value_arr[sl]
        sub_pop = pop_arr[sl]
        sub_valid = valid_val[sl] & sub_mask
        sub_weighted = sub_valid & has_pop_weight[sl]

        if not sub_weighted.any():
            continue

        unique_vals, pct_int = weighted_percentile_lookup(
            sub_value[sub_weighted], sub_pop[sub_weighted], invert=invert
        )

        out_sub = out[sl]
        out_sub[sub_valid] = assign_percentiles(sub_value[sub_valid], unique_vals, pct_int)
        out[sl] = out_sub

    return out

rel_dep_pctl = percentile_raster_by_country(
    value_arr, pop_arr, country_id_arr, valid_val, has_pop_weight, invert=True
)
save_raster(rel_dep_pctl, RESULTS_DIR / "relative_deprivation_percentiles_country.tif")

In [24]:
##### Step 2: global percentiles (same logic as step #1)

def percentile_raster_global(value_arr, pop_arr, valid_val, has_pop_weight, invert=False):
    out = np.full(value_arr.shape, np.nan, dtype=np.float32)
    weighted = valid_val & has_pop_weight

    unique_vals, pct_int = weighted_percentile_lookup(
        value_arr[weighted], pop_arr[weighted], invert=invert
    )
    out[valid_val] = assign_percentiles(value_arr[valid_val], unique_vals, pct_int)
    return out

global_rel_dep_pctl = percentile_raster_global(
    value_arr, pop_arr, valid_val, has_pop_weight, invert=True
)
save_raster(global_rel_dep_pctl, RESULTS_DIR / "relative_deprivation_percentiles_global.tif")

Country percentiles: 1,013,376 pixels
Global percentiles:  1,013,465 pixels
